# 📓 Notebook 03 — Mobile (Android / iOS)

Static APK analysis, hardcoded secrets, root/SSL-pinning bypass, exported components, dynamic instrumentation.

# 🧱 BASE (shared setup) — run these first, in order

In [ ]:
# Which GPU did you actually get? Decide the model from this.
!nvidia-smi
import torch, psutil
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM {p.total_memory/1e9:.1f} GB | RAM {psutil.virtual_memory().total/1e9:.1f} GB")
!df -h /content | tail -1

Wed Oct 14 07:43:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

GPU: Tesla T4 | VRAM 15.6 GB | RAM 13.6 GB
overlay         236G   77G  160G  33% /


In [ ]:
import os
try:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("cache:", CACHE)
except Exception as e:
    print("Drive notice:", e)
    CACHE = '/content/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("local cache:", CACHE)


cache: /content/drive/MyDrive/redteam


In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y docker.io git curl wget jq unzip tmux nodejs npm python3-venv python3-pip net-tools iputils-ping dnsutils
# Colab blocks Docker's iptables hooks -> disable them
dockerd --iptables=false --ip6tables=false --bridge=none >/var/log/dockerd.log 2>&1 &
for i in $(seq 1 30); do docker info >/dev/null 2>&1 && break; sleep 2; done
docker info | grep -E "Server Version|Storage Driver"
echo "docker ready" 

Server Version: 29.1.3-colab
Storage Driver: vfs
docker ready


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh || true
command -v ollama || { curl -L https://ollama.com/download/ollama-linux-amd64.tgz | tar -xz -C /usr; }
nohup ollama serve >/var/log/ollama.log 2>&1 &
for i in $(seq 1 40); do curl -s localhost:11434/api/version >/dev/null && break; sleep 2; done
curl -s localhost:11434/api/version && echo " <- ollama up" 

/usr/local/bin/ollama
{"version":"0.40.4"} <- ollama up


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
%%bash
if ollama list | grep -q "qwen2.5-coder:7b"; then
  echo "qwen2.5-coder:7b ready"
else
  ollama pull qwen2.5-coder:7b
fi
ollama list


qwen2.5-coder:7b ready
NAME                ID              SIZE      MODIFIED       
gpt-oss:20b         f38aa0c53da5    13 GB     22 minutes ago    
qwen2.5-coder:7b    dae161e27b0e    4.7 GB    43 minutes ago    


In [ ]:
%%bash
# HexStrike = 150+ tools behind one MCP endpoint
if [ ! -d /content/hexstrike ]; then
  git clone --depth 1 https://github.com/0x4m4/hexstrike-ai /content/hexstrike
fi
cd /content/hexstrike
if ! curl -sf localhost:8888/health >/dev/null; then
  nohup python3 hexstrike_server.py --port 8888 >/var/log/hexstrike.log 2>&1 &
  sleep 5
fi
echo "hexstrike:"; curl -s localhost:8888/health; echo

# Kali toolbox container the agent can shell into
docker pull -q kalilinux/kali-rolling || true
docker run -d --name redteam --network host kalilinux/kali-rolling sleep infinity || true
echo toolbox-ready


hexstrike:
{"all_essential_tools_available":false,"cache_stats":{"evictions":0,"hit_rate":"8.8%","hits":178,"max_size":1000,"misses":1854,"size":20},"category_stats":{"additional":{"available":1,"total":14},"api":{"available":1,"total":8},"binary":{"available":3,"total":13},"cloud":{"available":0,"total":10},"essential":{"available":4,"total":8},"exploitation":{"available":0,"total":3},"forensics":{"available":2,"total":16},"network":{"available":3,"total":10},"osint":{"available":1,"total":13},"password":{"available":0,"total":5},"vuln_scanning":{"available":1,"total":4},"web_security":{"available":4,"total":19},"wireless":{"available":0,"total":4}},"message":"HexStrike AI Tools API Server is operational","status":"healthy","telemetry":{"average_execution_time":"0.00s","commands_executed":1854,"success_rate":"1.1%","system_metrics":{"cpu_percent":5.6,"disk_usage":32.5,"memory_percent":28.1,"network_io":{"bytes_recv":32957995513,"bytes_sent":135292110,"dropin":0,"dropout":0,"errin":0,"

In [ ]:
%%bash
if ! pgrep -f "cloudflared tunnel --url http://localhost:8888" >/dev/null; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared 2>/dev/null || true
  chmod +x /usr/local/bin/cloudflared 2>/dev/null || true
  nohup cloudflared tunnel --url http://localhost:8888 >/var/log/cf.log 2>&1 &
  sleep 10
fi
echo "HexStrike URL:"; grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /var/log/cf.log | head -1


HexStrike URL:
https://modification-similarly-retention-theatre.trycloudflare.com


In [ ]:
%%bash
apt-get -qq install -y default-jdk android-tools-adb apktool aapt unzip
pip -q install frida-tools objection apkleaks 2>/dev/null || true
# jadx
JADX=1.5.0
wget -q https://github.com/skylot/jadx/releases/download/v${JADX}/jadx-${JADX}.zip -O /tmp/jadx.zip
unzip -q -o /tmp/jadx.zip -d /opt/jadx && ln -sf /opt/jadx/bin/jadx /usr/local/bin/jadx
# MobSF (web UI on :8000)
docker pull -q opensecurity/mobile-security-framework-mobsf
echo "mobile tools ready" 

Selecting previously unselected package android-liblog:amd64.
(Reading database ... 170267 files and directories currently installed.)
Preparing to unpack .../00-android-liblog_1%3a34.0.4-1build3_amd64.deb ...
Unpacking android-liblog:amd64 (1:34.0.4-1build3) ...
Selecting previously unselected package android-libbase:amd64.
Preparing to unpack .../01-android-libbase_1%3a34.0.4-1build3_amd64.deb ...
Unpacking android-libbase:amd64 (1:34.0.4-1build3) ...
Selecting previously unselected package android-libboringssl:amd64.
Preparing to unpack .../02-android-libboringssl_14.0.0+r11-4build1_amd64.deb ...
Unpacking android-libboringssl:amd64 (14.0.0+r11-4build1) ...
Selecting previously unselected package android-libcutils:amd64.
Preparing to unpack .../03-android-libcutils_1%3a34.0.4-1build3_amd64.deb ...
Unpacking android-libcutils:amd64 (1:34.0.4-1build3) ...
Selecting previously unselected package android-libziparchive:amd64.
Preparing to unpack .../04-android-libziparchive_1%3a34.0.4-1b

In [ ]:
%%bash
mkdir -p /content/apks
if [ ! -d /content/apks/diva ]; then
  git clone --depth 1 https://github.com/payatu/diva-android /content/apks/diva 2>/dev/null || true
fi
if [ ! -d /content/apks/AndroGoat ]; then
  git clone --depth 1 https://github.com/satishpatnayak/AndroGoat /content/apks/AndroGoat 2>/dev/null || true
fi
find /content/apks -name "*.apk" | head


In [ ]:
%%bash
docker run -d --name mobsf --network host opensecurity/mobile-security-framework-mobsf
sleep 25
echo "MobSF -> http://localhost:8000  (user: mobsf / pass: mobsf)"
curl -s -o /dev/null -w "mobsf:%{http_code}\n" http://localhost:8000

Container mobsf started
MobSF -> http://localhost:8000  (user: mobsf / pass: mobsf)
mobsf:000


CalledProcessError: Command 'b'docker run -d --name mobsf --network host opensecurity/mobile-security-framework-mobsf\nsleep 25\necho "MobSF -> http://localhost:8000  (user: mobsf / pass: mobsf)"\ncurl -s -o /dev/null -w "mobsf:%{http_code}\\n" http://localhost:8000\n'' returned non-zero exit status 7.

In [ ]:
# --- static analysis without a device ---
import glob, os
apks = glob.glob('/content/apks/**/*.apk', recursive=True)
apk = apks[0] if apks else '/content/apks/diva-beta.apk'
print(f'Analyzing APK: {apk}')
!jadx -d /content/diva_src {apk} 2>/dev/null || true
!apktool d -f -o /content/diva_apktool {apk} 2>/dev/null || true
!grep -rEi "api[_-]?key|secret|password|token" /content/diva_src 2>/dev/null | head -20 || true
!strings {apk} 2>/dev/null | grep -Ei "http|firebase|amazonaws" | sort -u | head || true


Analyzing APK: /content/apks/diva-beta.apk


[0.016s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.016s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


ERROR - Incorrect arguments: File not found /content/apks/diva-beta.apk


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [ ]:
%%bash
# on the VM
nohup frida-server >/dev/null 2>&1 &   # or push frida-server to device
echo "adb devices"; adb devices || true
echo "frida-ps -U"; frida-ps -U 2>/dev/null | head || true

adb devices
List of devices attached

frida-ps -U


* daemon not running; starting now at tcp:5037
* daemon started successfully
